# 02_feature_engineering.ipynb
## Feature Engineering for Web Traffic Forecasting

This notebook loads the cleaned and sampled dataset produced in Notebook 01
and creates:

- Lag features
- Rolling mean and std features
- Date-based features (day of week, month)
- Encoded metadata (language, access type, agent)
- Train/validation split

Output: `features.csv` for modeling.


## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder


## Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Load the cleaned dataset

This file is created from Notebook 01 and contains:
- Log-transformed values
- Sampled pages
- Cleaned missing values


In [3]:
df = pd.read_csv("/content/drive/MyDrive/CSC480/Final Project/cleaned_sample.csv")
df.head()

,Page,2015-07-01,2015-07-02,2015-07-03,2015-07-04,2015-07-05,2015-07-06,2015-07-07,2015-07-08,2015-07-09,...,2016-12-25,2016-12-26,2016-12-27,2016-12-28,2016-12-29,2016-12-30,2016-12-31,language,access,agent
0,Phabricator/Project_management_www.mediawiki.o...,1.945910,1.945910,1.609438,1.945910,2.197225,1.945910,1.609438,0.000000,1.098612,...,1.609438,1.945910,1.791759,2.079442,1.945910,1.945910,2.302585,Phabricator/Project_management_www,unknown,spider
1,Now_You_See_Me_es.wikipedia.org_desktop_all-ag...,5.493061,5.605802,5.736572,5.429346,5.774552,5.743003,5.493061,5.468060,5.497168,...,5.438079,5.814131,5.758902,5.783825,5.594711,5.308268,5.252273,Now_You_See_Me_es,desktop,user
2,Zürich_Hackathon_2014_www.mediawiki.org_all-ac...,1.386294,2.995732,2.995732,3.433987,3.091042,3.218876,2.890372,5.187386,3.713572,...,2.197225,1.098612,1.609438,2.302585,1.609438,2.484907,2.564949,Zürich_Hackathon_2014_www,unknown,spider
3,Érythrée_fr.wikipedia.org_desktop_all-agents,6.511745,6.242223,6.652863,7.060476,6.304449,6.628041,6.320768,6.204558,8.476788,...,5.323010,5.780744,6.084499,5.846439,5.703782,5.726848,5.356586,Érythrée_fr,desktop,user
4,Metallica_es.wikipedia.org_all-access_all-agents,7.336286,7.405496,7.441320,7.358831,7.336286,7.363914,7.383368,7.457032,7.560080,...,7.635787,7.928406,7.845024,7.833996,7.765993,7.737180,7.675082,Metallica_es,unknown,user


## Identify date columns

All date columns represent daily log-transformed page views.

We will generate lag and rolling features from these.


In [4]:
# metadata columns
meta_cols = ["Page", "language", "access", "agent"]

# date columns
date_cols = [c for c in df.columns if c not in meta_cols]

print("Number of dates:", len(date_cols))
print("Total rows:", df.shape[0])


Number of dates: 550
Total rows: 500


## Create Lag Features

We generate lag features of:
- 7 days
- 14 days
- 30 days

These capture short-term temporal dependencies.


In [5]:
df_lag = df.copy()

LAGS = [7, 14, 30]

for lag in LAGS:
    shifted = df_lag[date_cols].shift(lag, axis=1)
    shifted.columns = [f"{c}_lag{lag}" for c in date_cols]
    df_lag = pd.concat([df_lag, shifted], axis=1)

df_lag.head()



,Page,2015-07-01,2015-07-02,2015-07-03,2015-07-04,2015-07-05,2015-07-06,2015-07-07,2015-07-08,2015-07-09,...,2016-12-22_lag30,2016-12-23_lag30,2016-12-24_lag30,2016-12-25_lag30,2016-12-26_lag30,2016-12-27_lag30,2016-12-28_lag30,2016-12-29_lag30,2016-12-30_lag30,2016-12-31_lag30
0,Phabricator/Project_management_www.mediawiki.o...,1.945910,1.945910,1.609438,1.945910,2.197225,1.945910,1.609438,0.000000,1.098612,...,2.079442,2.995732,2.397895,1.386294,2.079442,2.639057,2.484907,2.397895,2.079442,2.397895
1,Now_You_See_Me_es.wikipedia.org_desktop_all-ag...,5.493061,5.605802,5.736572,5.429346,5.774552,5.743003,5.493061,5.468060,5.497168,...,6.061457,6.572283,5.976351,5.820083,5.958425,6.163315,6.045005,5.758902,5.834811,5.631212
2,Zürich_Hackathon_2014_www.mediawiki.org_all-ac...,1.386294,2.995732,2.995732,3.433987,3.091042,3.218876,2.890372,5.187386,3.713572,...,1.609438,1.945910,2.197225,2.079442,2.484907,2.197225,1.791759,1.791759,1.791759,1.791759
3,Érythrée_fr.wikipedia.org_desktop_all-agents,6.511745,6.242223,6.652863,7.060476,6.304449,6.628041,6.320768,6.204558,8.476788,...,6.146329,6.139885,6.216606,5.971262,5.852202,5.852202,6.133398,6.073045,5.983936,5.950643
4,Metallica_es.wikipedia.org_all-access_all-agents,7.336286,7.405496,7.441320,7.358831,7.336286,7.363914,7.383368,7.457032,7.560080,...,8.469053,8.542081,8.502689,8.325791,8.250881,8.264878,8.151333,8.151045,8.060224,8.022241


## Rolling Features

Rolling mean and std provide smoothed representations of trends.
Window sizes: 7-day rolling.


In [6]:
WINDOW = 7

# rolling mean across all date columns
roll_mean = df_lag[date_cols].rolling(WINDOW, axis=1).mean()
roll_mean.columns = [f"{c}_roll_mean" for c in date_cols]

# rolling std across all date columns
roll_std = df_lag[date_cols].rolling(WINDOW, axis=1).std()
roll_std.columns = [f"{c}_roll_std" for c in date_cols]

# add them to the dataframe
df_lag = pd.concat([df_lag, roll_mean, roll_std], axis=1)

df_lag.head()



/tmp/ipython-input-345540429.py:4: FutureWarning: Support for axis=1 in DataFrame.rolling is deprecated and will be removed in a future version. Use obj.T.rolling(...) instead
  roll_mean = df_lag[date_cols].rolling(WINDOW, axis=1).mean()
/tmp/ipython-input-345540429.py:8: FutureWarning: Support for axis=1 in DataFrame.rolling is deprecated and will be removed in a future version. Use obj.T.rolling(...) instead
  roll_std = df_lag[date_cols].rolling(WINDOW, axis=1).std()


,Page,2015-07-01,2015-07-02,2015-07-03,2015-07-04,2015-07-05,2015-07-06,2015-07-07,2015-07-08,2015-07-09,...,2016-12-22_roll_std,2016-12-23_roll_std,2016-12-24_roll_std,2016-12-25_roll_std,2016-12-26_roll_std,2016-12-27_roll_std,2016-12-28_roll_std,2016-12-29_roll_std,2016-12-30_roll_std,2016-12-31_roll_std
0,Phabricator/Project_management_www.mediawiki.o...,1.945910,1.945910,1.609438,1.945910,2.197225,1.945910,1.609438,0.000000,1.098612,...,0.460600,0.462763,0.428527,0.506236,0.514896,0.534076,0.270985,0.270985,0.270985,0.216804
1,Now_You_See_Me_es.wikipedia.org_desktop_all-ag...,5.493061,5.605802,5.736572,5.429346,5.774552,5.743003,5.493061,5.468060,5.497168,...,0.130555,0.152525,0.199853,0.169041,0.190357,0.196959,0.220090,0.214486,0.230481,0.233897
2,Zürich_Hackathon_2014_www.mediawiki.org_all-ac...,1.386294,2.995732,2.995732,3.433987,3.091042,3.218876,2.890372,5.187386,3.713572,...,0.440639,0.441192,0.294580,0.259850,0.429010,0.440246,0.421489,0.425851,0.494383,0.547694
3,Érythrée_fr.wikipedia.org_desktop_all-agents,6.511745,6.242223,6.652863,7.060476,6.304449,6.628041,6.320768,6.204558,8.476788,...,0.105673,0.107067,0.114482,0.196998,0.198785,0.233202,0.232868,0.233754,0.232204,0.269482
4,Metallica_es.wikipedia.org_all-access_all-agents,7.336286,7.405496,7.441320,7.358831,7.336286,7.363914,7.383368,7.457032,7.560080,...,0.066873,0.044610,0.025731,0.046309,0.089099,0.095800,0.098498,0.098561,0.097406,0.102299


## Encode the metadata

We convert:
- language
- access
- agent

into numeric form using label encoding.


In [7]:
le_lang = LabelEncoder()
le_acc = LabelEncoder()
le_agent = LabelEncoder()

df_lag["lang_enc"] = le_lang.fit_transform(df_lag["language"])
df_lag["access_enc"] = le_acc.fit_transform(df_lag["access"])
df_lag["agent_enc"] = le_agent.fit_transform(df_lag["agent"])

df_lag.head()


,Page,2015-07-01,2015-07-02,2015-07-03,2015-07-04,2015-07-05,2015-07-06,2015-07-07,2015-07-08,2015-07-09,...,2016-12-25_roll_std,2016-12-26_roll_std,2016-12-27_roll_std,2016-12-28_roll_std,2016-12-29_roll_std,2016-12-30_roll_std,2016-12-31_roll_std,lang_enc,access_enc,agent_enc
0,Phabricator/Project_management_www.mediawiki.o...,1.945910,1.945910,1.609438,1.945910,2.197225,1.945910,1.609438,0.000000,1.098612,...,0.506236,0.514896,0.534076,0.270985,0.270985,0.270985,0.216804,248,2,0
1,Now_You_See_Me_es.wikipedia.org_desktop_all-ag...,5.493061,5.605802,5.736572,5.429346,5.774552,5.743003,5.493061,5.468060,5.497168,...,0.169041,0.190357,0.196959,0.220090,0.214486,0.230481,0.233897,236,0,1
2,Zürich_Hackathon_2014_www.mediawiki.org_all-ac...,1.386294,2.995732,2.995732,3.433987,3.091042,3.218876,2.890372,5.187386,3.713572,...,0.259850,0.429010,0.440246,0.421489,0.425851,0.494383,0.547694,337,2,0
3,Érythrée_fr.wikipedia.org_desktop_all-agents,6.511745,6.242223,6.652863,7.060476,6.304449,6.628041,6.320768,6.204558,8.476788,...,0.196998,0.198785,0.233202,0.232868,0.233754,0.232204,0.269482,341,0,1
4,Metallica_es.wikipedia.org_all-access_all-agents,7.336286,7.405496,7.441320,7.358831,7.336286,7.363914,7.383368,7.457032,7.560080,...,0.046309,0.089099,0.095800,0.098498,0.098561,0.097406,0.102299,223,2,1


## Create Train/Validation Split

We forecast the next **60 days**.

- Training data: all except last 60 dates  
- Validation data: last 60 dates  


In [8]:
FORECAST_HORIZON = 60

train_dates = date_cols[:-FORECAST_HORIZON]
val_dates = date_cols[-FORECAST_HORIZON:]

print("Training days:", len(train_dates))
print("Validation days:", len(val_dates))


Training days: 490
Validation days: 60


## Prepare train and validation datasets

We reshape:
- X = features from lag + metadata  
- y = actual future values  


In [9]:
# Features will be the last valid lag feature of each row
feature_cols = [c for c in df_lag.columns if "lag" in c or "roll" in c] + ["lang_enc", "access_enc", "agent_enc"]

X = df_lag[feature_cols]
y = df_lag[val_dates]  # next 60 days

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (500, 2753)
y shape: (500, 60)


## Save the engineered features

These files will be used in:
- lightgbm.ipynb  
- lstm.ipynb  
- ensemble.ipynb  


In [10]:
save_dir = "/content/drive/MyDrive/CSC480/Final Project/"
df_lag.to_csv(save_dir + "features.csv", index=False)
y.to_csv(save_dir + "labels.csv", index=False)

print("Saved features and labels.")


Saved features and labels.
